In [3]:
import os
import shutil
from MRE import Job # Importamos el emulador

# Etiquetas para identificar de qué dataset viene cada equipo
TAG_ESTANDAR = "EST"
TAG_PLATINIUM = "PLAT"

# --- FASE MAP ---
# Se ejecuta por cada línea. key = ID local, value = "ID_visitante ap_local ap_vis"
# Hay un map definido por dataset para etiquetar el origen (EST / PLAT).
# A diferencia del punto 1, acá se FILTRA en el map: solo emitimos al visitante
# si en ese partido recibió más apuestas que el local.

# Filtramos el dataset Estandar
def fmapEstandar(key, value, context):
    datos = value.split() # separa la línea en campos
    id_local = key  # ID_equipo_local es la clave
    id_visitante = datos[0]  # ID_equipo_visitante
    ap_local = int(datos[1])  # usuarios que apostaron al local como ganador
    ap_visitante = int(datos[2])  # usuarios que apostaron al visitante como ganador

    # Solo emitimos si la cantidad de apuestas del visitante superó a las apuestas del local en ese partido
    if ap_visitante > ap_local:
        context.write(id_visitante, TAG_ESTANDAR)

# Filtramos el dataset Platinium
def fmapPlatinium(key, value, context):
    datos = value.split() # separa la línea en campos
    id_local = key  # ID_equipo_local es la clave
    id_visitante = datos[0]  # ID_equipo_visitante
    ap_local = int(datos[1])  # usuarios que apostaron al local como ganador
    ap_visitante = int(datos[2])  # usuarios que apostaron al visitante como ganador

    # Solo emitimos si la cantidad de apuestas del visitante superó a las apuestas del local en ese partido
    if ap_visitante > ap_local:
        context.write(id_visitante, TAG_PLATINIUM)

# --- FASE COMBINER ---
# Elimina etiquetas repetidas por equipo, solo con lo que vio su mapper.
# Reduce el tráfico de red. NO exige ambas etiquetas (ya que solo ve datos parciales).
def fcombiner(key, values, context):
    # set() deja una sola vez cada etiqueta; emitimos el mismo formato que el map
    for tag in set(values):
        context.write(key, tag)

# --- FASE REDUCE ---
# Recibe todas las etiquetas de UN equipo (key) ya agrupadas por el shuffle.
# Acá se decide la intersección.
def fred(key, values, context):
    tags = set(values) # etiquetas distintas que llegaron para este equipo

    # Intersección: el equipo debe cumplir la condición en AMBOS datasets
    # (no necesariamente en el mismo partido)
    if TAG_ESTANDAR in tags and TAG_PLATINIUM in tags:
        context.write(key, "") # Si cumple ambas condiciones lo imprimimos

# --- DRIVER ---
# Configura y dispara el job
if __name__ == '__main__':
    input_est = "/content/in_estandar"     # carpeta con el dataset estándar
    input_plat = "/content/in_platinium"   # carpeta con el dataset platinium
    output_dir = "/content/salida_punto2"  # carpeta de resultados

    # Preparación: crear carpetas de entrada y copiar los archivos
    os.makedirs(input_est, exist_ok=True)
    os.makedirs(input_plat, exist_ok=True)
    shutil.copy("/content/apuestasEstandar.txt", input_est + "/apuestasEstandar.txt")
    shutil.copy("/content/apuestasPlatinium.txt", input_plat + "/apuestasPlatinium.txt")

    # Borrar la salida anterior (si existe) para que no falle al re-ejecutar
    shutil.rmtree(output_dir, ignore_errors=True)

    job2 = Job(input_est, output_dir, fmapEstandar, fred) # Job con el primer dataset estándar y su map correspondiente
    job2.addInputPath(input_plat, fmapPlatinium) # Segundo dataset platinium con su map
    job2.setCombiner(fcombiner) # Combiner para reducir datos enviados por la red

    success = job2.waitForCompletion()

    if success:
        print("El job finalizó correctamente!")

El job finalizó correctamente!
